# Regularization: Ridge & Lasso

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/linear-regression/03-regularization

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Intuition — penalize complexity to generalize

With many features (especially correlated or noisy ones), plain least squares **overfits**: it
fits the training noise with large, unstable coefficients that generalize poorly. **Regularization**
adds a penalty on the weights to the loss, trading a little training fit for smaller, more robust
coefficients. **Ridge (L2)** penalizes `Σwᵢ²` — it shrinks all weights smoothly toward 0 and tames
correlated features. **Lasso (L1)** penalizes `Σ|wᵢ|` — its corners drive many weights to *exactly*
0, performing automatic **feature selection**. We build both from scratch (Ridge in closed form,
Lasso via proximal gradient / ISTA), watch their regularization paths, and validate against
`sklearn`.

## Overfitting with correlated features

We build a dataset where only 3 of 20 features matter, then watch OLS, Ridge, and Lasso handle it.

In [ ]:
n, d = 60, 20
X = np.random.randn(n, d)
true_w = np.zeros(d); true_w[:3] = [3.0, -2.0, 1.5]
y = X @ true_w + 0.5 * np.random.randn(n)

# closed-form ridge: w = (X'X + lam*I)^-1 X'y   (lam=0 -> OLS)
def ridge(X, y, lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

print("OLS weights (first 6):", np.round(ridge(X, y, 0)[:6], 2))
print("Ridge λ=10  (first 6):", np.round(ridge(X, y, 10)[:6], 2))

**What to notice:** the data has only **3 true features** (weights 3, −2, 1.5) among 20. OLS
spreads nonzero weight across the noise features too; **Ridge (λ=10)** pulls all of them toward 0,
noticeably shrinking the spurious ones. Ridge doesn't zero them, but it stops them from running wild.

## Lasso via proximal gradient (ISTA)

The L1 penalty has no closed form, but coordinate-wise soft-thresholding solves it.

In [ ]:
def soft(z, t): return np.sign(z) * np.maximum(np.abs(z) - t, 0)

def lasso(X, y, lam, iters=500):
    w = np.zeros(X.shape[1]); L = np.linalg.norm(X, 2) ** 2
    for _ in range(iters):
        w = soft(w - X.T @ (X @ w - y) / L, lam / L)
    return w

w_lasso = lasso(X, y, lam=15)
print("Lasso non-zero weights:", np.flatnonzero(np.abs(w_lasso) > 1e-6))
print("values:", np.round(w_lasso[np.abs(w_lasso) > 1e-6], 2))

**What to notice:** **Lasso** keeps only features `[0, 1, 2]` — the true ones — and sets the
other 17 to **exactly 0**. That's the qualitative difference from Ridge: L1's penalty has corners at
0, so the optimum lands *on* the axes, producing a **sparse** model that doubles as feature
selection.

## Regularization paths

Watch every weight as λ sweeps — Ridge shrinks smoothly, Lasso snaps weights to exactly zero.

In [ ]:
lams = np.logspace(-2, 3, 40)
ridge_path = np.array([ridge(X, y, l) for l in lams])
lasso_path = np.array([lasso(X, y, l) for l in lams])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, path, name in [(axes[0], ridge_path, 'Ridge'), (axes[1], lasso_path, 'Lasso')]:
    for j in range(d):
        ax.plot(lams, path[:, j], color='#6366f1' if j < 3 else '#475569', lw=1.5 if j < 3 else 0.7)
    ax.set_xscale('log'); ax.set_xlabel('λ'); ax.set_title(f'{name} path')
axes[0].set_ylabel('weight value')
plt.tight_layout(); plt.show()
# purple = the 3 true features; gray = the 17 noise features

**What to notice:** in the paths, the 3 purple (true) weights stay large while the gray (noise)
weights decay as `λ` grows. **Ridge** shrinks them smoothly toward 0; **Lasso** snaps them to
**exactly 0** at finite `λ` (they hit the axis and stay). The Lasso plot visibly has weights
vanishing one by one — the geometry of sparsity.

## The library way — validate against `sklearn`

Our closed-form Ridge should match `sklearn.Ridge`, and our ISTA Lasso should select the same true
features as `sklearn.Lasso`. The cell asserts both (Lasso's `alpha` uses a `1/n` scaling relative to
our `λ`, so we compare the *selected support* rather than raw magnitudes).

In [ ]:
from sklearn.linear_model import Ridge as SkRidge, Lasso as SkLasso

# Ridge: exact match (both penalize every coefficient, no intercept column here)
lam = 10.0
assert np.allclose(ridge(X, y, lam), SkRidge(alpha=lam, fit_intercept=False).fit(X, y).coef_), \
    "closed-form Ridge must match sklearn"

# Lasso: both should pick out the 3 true features as the top coefficients
sk_lasso = SkLasso(alpha=0.1, fit_intercept=False, max_iter=10000).fit(X, y)
top_ours = set(np.argsort(np.abs(w_lasso))[-3:])
top_sk   = set(np.argsort(np.abs(sk_lasso.coef_))[-3:])
print('true features : {0, 1, 2}')
print('our Lasso top-3    :', sorted(top_ours))
print('sklearn Lasso top-3:', sorted(top_sk))
assert top_ours == {0, 1, 2} and top_sk == {0, 1, 2}, "both must recover the true support"
print('\nRidge == sklearn.Ridge, and both Lassos select the true features ✓')

**What to notice:** our closed-form Ridge equals `sklearn`'s exactly, and both Lasso
implementations recover the same 3 true features out of 20. The from-scratch proximal-gradient
Lasso does the real thing — `sklearn` just uses a faster coordinate-descent solver.

## Gotchas & tradeoffs

- **Standardize features first.** L1/L2 penalize by coefficient magnitude, so unscaled features are
  penalized unequally — always standardize before regularizing.
- **λ is a hyperparameter.** Too small → overfit, too large → underfit (everything shrinks to 0).
  Choose it by cross-validation (`RidgeCV`/`LassoCV`).
- **Lasso is arbitrary among correlated features.** It tends to pick *one* of a correlated group and
  zero the rest; **Elastic Net** (L1+L2) is more stable there.
- **L1 isn't differentiable at 0.** That's why Lasso needs proximal/coordinate methods (soft
  thresholding), not plain gradient descent.

In [ ]:
# lambda controls the bias-variance tradeoff: too big shrinks everything to ~0
for lam in [0.0, 1.0, 100.0, 1000.0]:
    w = ridge(X, y, lam)
    print(f'lambda={lam:>7}: ||w|| = {np.linalg.norm(w):5.2f}, '
          f'weight on true feature 0 = {w[0]:.2f} (true 3.0)')
print('\n-> small lambda keeps the signal; huge lambda crushes even the true weights toward 0')

**What to notice:** at `λ=0` (OLS) the true weight is recovered (~3.0), but as `λ` grows the norm
shrinks and even the *real* signal gets crushed toward 0 by `λ=1000`. Regularization strength is a
dial between overfitting (too little) and underfitting (too much) — cross-validation finds the
sweet spot.

## Key takeaways

- **Regularization** adds a weight penalty to fight overfitting, trading training fit for
  generalization.
- **Ridge (L2)** shrinks all weights smoothly (great for correlated features); **Lasso (L1)** zeros
  many exactly → **sparse feature selection**.
- Lasso needs proximal/coordinate methods (soft-thresholding) because L1 has a kink at 0;
  **Elastic Net** blends L1+L2.
- **Standardize** features, and pick **λ by cross-validation**. Both match `sklearn` when set up
  consistently.

**Next:** [Generalized Linear Models](https://ml-viz-ruby.vercel.app/courses/linear-regression/04-generalized-linear-models).

**Try it:** raise the noise level to 2.0, or make features correlated with `X[:, 3] = X[:, 0] + 0.01*np.random.randn(n)`. Watch what OLS does to those two weights vs. Ridge.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Soft thresholding

The operator at the heart of lasso (and of ISTA above) shrinks every value toward zero by $t$ — and snaps anything within $t$ of zero to **exactly zero**:

$$S_t(x) = \text{sign}(x) \cdot \max(|x| - t, \, 0)$$

That hard zero is where lasso's sparsity comes from (ridge only ever scales weights down — it never zeroes them). Implement it vectorized.

In [ ]:
def soft_threshold(x, t):
    """Shrink x toward 0 by t; values within t of 0 become exactly 0."""
    x = np.asarray(x, dtype=float)

    # TODO(you): sign(x) * max(|x| - t, 0) — keep it vectorized
    # (hint: np.sign, np.maximum, np.abs)
    return ...

In [ ]:
# Checks — run me
assert np.allclose(soft_threshold(3.0, 1.0), 2.0), "shrink positive values toward 0"
assert np.allclose(soft_threshold(-3.0, 1.0), -2.0), "shrink negative values toward 0"
assert np.allclose(soft_threshold(0.4, 1.0), 0.0), "inside the threshold -> exactly 0 (sparsity!)"
assert np.allclose(soft_threshold(np.array([-2.0, -0.5, 0.0, 0.5, 2.0]), 1.0),
                   [-1.0, 0.0, 0.0, 0.0, 1.0]), "works on arrays"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def soft_threshold(x, t):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.maximum(np.abs(x) - t, 0.0)
```

</details>

### Exercise 2 — One ISTA step

ISTA alternates a plain gradient step on the squared error with a soft-threshold whose level is $\eta\lambda$:

$$\mathbf{w} \leftarrow S_{\eta\lambda}\!\big(\mathbf{w} - \eta \, X^\top(X\mathbf{w} - \mathbf{y})\big)$$

Implement one step. With an **identity design matrix** ($X = I$) and $\eta = 1$, a single step from $\mathbf{w} = 0$ lands *exactly* on the lasso solution $S_\lambda(\mathbf{y})$ — the checks verify that, the exact zero it creates, and that iterating from elsewhere converges to the same point.

In [ ]:
def ista_step(w, X, y, lam, lr):
    """One ISTA update for the lasso objective 0.5*||Xw - y||^2 + lam*||w||_1."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): the squared-error gradient X^T (X w - y)
    grad = ...

    # TODO(you): gradient step, then soft-threshold at level lr * lam
    return ...

In [ ]:
# Checks — run me
y_t = np.array([3.0, -0.5, 1.5])
I = np.eye(3)

w1 = ista_step(np.zeros(3), I, y_t, lam=1.0, lr=1.0)
assert np.allclose(w1, soft_threshold(y_t, 1.0)), \
    "identity design, lr = 1: one step from 0 lands exactly on soft_threshold(y, lam)"
assert w1[1] == 0.0, "the small coefficient is zeroed exactly — lasso sparsity"

w = np.array([10.0, -10.0, 10.0])
for _ in range(50):
    w = ista_step(w, I, y_t, lam=1.0, lr=0.5)
assert np.allclose(w, soft_threshold(y_t, 1.0), atol=1e-8), "ISTA converges to the lasso solution"
# Edge case: near-singular / multicollinear design matrix (two identical columns -- the exact
# failure mode the 'Try it' note above hints at). The per-column weights are not unique here,
# so check the *fitted values* converge well instead of any single coefficient.
X_dup = np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0], [4.0, 4.0]])
y_dup = np.array([2.0, 4.0, 6.0, 8.0])
L_dup = np.linalg.norm(X_dup, 2) ** 2
w_dup = np.zeros(2)
for _ in range(300):
    w_dup = ista_step(w_dup, X_dup, y_dup, lam=0.01, lr=1.0 / L_dup)
assert np.all(np.isfinite(w_dup)), "must stay finite under exact multicollinearity"
assert np.allclose(X_dup @ w_dup, y_dup, atol=1e-1), \
    "predictions still converge to y even though the two identical columns split the weight arbitrarily"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ista_step(w, X, y, lam, lr):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    grad = X.T @ (X @ w - y)
    return soft_threshold(w - lr * grad, lr * lam)
```

</details>

---
## 🌐 Extra practice — from Open-Deep-ML

Three [DML](https://github.com/Open-Deep-ML/DML-OpenProblem) drills that build up to a capstone: the Ridge loss as its own function, Lasso trained with plain (sub-)gradient descent instead of ISTA, and finally Elastic Net — L1 and L2 together.

### Exercise 3 — Ridge loss function (DML #43)

So far Ridge has meant the closed-form weights. DML #43 asks for the *objective itself*: $\text{MSE} + \alpha\lVert w \rVert^2$, evaluated at a given `w` (no fitting involved). This is the quantity the closed form in section 1 actually minimizes.

In [ ]:
def ridge_loss(X, w, y_true, alpha):
    """DML #43 signature: evaluate the Ridge objective at a given w."""
    X = np.asarray(X, dtype=float)
    w = np.asarray(w, dtype=float)
    y_true = np.asarray(y_true, dtype=float)

    # TODO(you): mean((y_true - X @ w)**2) + alpha * sum(w**2)
    return ...

In [ ]:
# Checks — run me
loss1 = ridge_loss(np.array([[1, 1], [2, 1], [3, 1], [4, 1]]), np.array([.2, 2]), np.array([2, 3, 4, 5]), 0.1)
assert round(loss1, 3) == 2.204, "DML #43 example"

loss2 = ridge_loss(np.array([[1, 1, 4], [2, 1, 2], [3, 1, .1], [4, 1, 1.2], [1, 2, 3]]),
                   np.array([.2, 2, 5]), np.array([2, 3, 4, 5, 2]), 0.1)
assert round(loss2, 3) == 164.402, "DML #43 second test case"

# alpha = 0 must reduce to plain MSE
assert np.isclose(ridge_loss(np.array([[1, 1], [2, 1]]), np.array([1.0, 0.0]), np.array([1.0, 2.0]), 0.0),
                  np.mean((np.array([1.0, 2.0]) - np.array([[1, 1], [2, 1]]) @ np.array([1.0, 0.0])) ** 2))
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ridge_loss(X, w, y_true, alpha):
    X = np.asarray(X, dtype=float)
    w = np.asarray(w, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    return np.mean((y_true - X @ w) ** 2) + alpha * np.sum(w ** 2)
```

</details>

### Exercise 4 — Lasso by (sub-)gradient descent (DML #50)

Above, Lasso was solved with ISTA (proximal gradient). DML #50 takes a more direct route: plain gradient descent using `sign(w)` as the L1 subgradient, applied directly to the weight update (not through a separate soft-threshold step) — and it fits an **unpenalized bias** term alongside the weights, which the earlier sections didn't need since they worked with a bias column baked into `X`.

In [ ]:
def l1_regularization_gradient_descent(X, y, alpha=0.1, learning_rate=0.01, max_iter=1000, tol=1e-4):
    """DML #50 signature: plain GD with an L1 subgradient penalty, separate bias term."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0.0

    for _ in range(max_iter):
        y_pred = X @ weights + bias
        error = y_pred - y
        # TODO(you): grad_w = mean-scaled gradient of the MSE term + alpha * sign(weights)
        # (hint: (X.T @ error) / n_samples + alpha * np.sign(weights))
        grad_w = ...
        # TODO(you): grad_b = mean-scaled gradient of the MSE term, NO penalty on the bias
        grad_b = ...

        weights -= learning_rate * grad_w
        bias -= learning_rate * grad_b

        if np.linalg.norm(grad_w, ord=1) < tol:
            break

    return weights, bias

In [ ]:
# Checks — run me
w1, b1 = l1_regularization_gradient_descent(np.array([[0, 0], [1, 1], [2, 2]]), np.array([0, 1, 2]),
                                            alpha=0.1, learning_rate=0.01, max_iter=1000)
assert np.allclose(w1, [0.4237, 0.4237], atol=1e-3), "DML #50 example 1 weights"
assert np.isclose(b1, 0.1539, atol=1e-3), "DML #50 example 1 bias"

w2, b2 = l1_regularization_gradient_descent(np.array([[0, 1], [1, 2], [2, 3], [3, 4], [4, 5]]),
                                            np.array([1, 2, 3, 4, 5]), alpha=0.1, learning_rate=0.01, max_iter=1000)
assert np.allclose(w2, [0.2728, 0.6811], atol=1e-3), "DML #50 example 2 weights"

# Edge case: near-singular design matrix (duplicate columns, like the ISTA edge case above)
w_dup2, b_dup2 = l1_regularization_gradient_descent(np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]),
                                                    np.array([2.0, 4.0, 6.0]), alpha=0.01,
                                                    learning_rate=0.01, max_iter=2000)
assert np.all(np.isfinite(w_dup2)) and np.isfinite(b_dup2), "must stay finite under collinearity"
assert np.allclose(np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]) @ w_dup2 + b_dup2,
                   [2.0, 4.0, 6.0], atol=0.5), "fitted values should still track y reasonably"
print("✅ Exercise 4 passed")

<details>
<summary>💡 Show solution</summary>

```python
def l1_regularization_gradient_descent(X, y, alpha=0.1, learning_rate=0.01, max_iter=1000, tol=1e-4):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0.0

    for _ in range(max_iter):
        y_pred = X @ weights + bias
        error = y_pred - y
        grad_w = (X.T @ error) / n_samples + alpha * np.sign(weights)
        grad_b = np.sum(error) / n_samples

        weights -= learning_rate * grad_w
        bias -= learning_rate * grad_b

        if np.linalg.norm(grad_w, ord=1) < tol:
            break

    return weights, bias
```

</details>

### Exercise 5 — Elastic Net: L1 + L2 together (DML #139)

The capstone: combine Exercise 4's L1 subgradient with Ridge's L2 gradient in one update. Elastic Net keeps Lasso's sparsity while handling groups of correlated features more gracefully than pure L1 (which tends to arbitrarily pick just one feature from a correlated group).

In [ ]:
def elastic_net_gradient_descent(X, y, alpha1=0.1, alpha2=0.1, learning_rate=0.01, max_iter=1000, tol=1e-4):
    """DML #139 signature: alpha1 scales the L1 term, alpha2 scales the L2 term."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0.0

    for _ in range(max_iter):
        y_pred = X @ weights + bias
        error = y_pred - y
        # TODO(you): grad_w = MSE gradient + alpha1 * sign(weights) + 2 * alpha2 * weights
        grad_w = ...
        grad_b = np.sum(error) / n_samples

        weights -= learning_rate * grad_w
        bias -= learning_rate * grad_b

        if np.linalg.norm(grad_w, ord=1) < tol:
            break

    return weights, bias

In [ ]:
# Checks — run me
w1, b1 = elastic_net_gradient_descent(np.array([[0, 0], [1, 1], [2, 2]]), np.array([0, 1, 2]),
                                     alpha1=0.1, alpha2=0.1, learning_rate=0.01, max_iter=1000)
assert np.allclose(np.round(w1, 2), [0.37, 0.37]), "DML #139 example 1 weights"
assert round(b1, 2) == 0.25, "DML #139 example 1 bias"

w2, b2 = elastic_net_gradient_descent(np.array([[0, 1], [1, 2], [2, 3], [3, 4], [4, 5]]),
                                     np.array([1, 2, 3, 4, 5]), alpha1=0.1, alpha2=0.1,
                                     learning_rate=0.01, max_iter=2000)
assert np.allclose(np.round(w2, 2), [0.43, 0.48]), "DML #139 example 2 weights"
assert round(b2, 2) == 0.69, "DML #139 example 2 bias"

# alpha2 = 0 should behave like the pure-Lasso Exercise 4 (same objective, same update rule)
w_lasso_like, b_lasso_like = elastic_net_gradient_descent(np.array([[0, 0], [1, 1], [2, 2]]), np.array([0, 1, 2]),
                                                          alpha1=0.1, alpha2=0.0, learning_rate=0.01, max_iter=1000)
w_pure_lasso, b_pure_lasso = l1_regularization_gradient_descent(np.array([[0, 0], [1, 1], [2, 2]]),
                                                                np.array([0, 1, 2]), alpha=0.1,
                                                                learning_rate=0.01, max_iter=1000)
assert np.allclose(w_lasso_like, w_pure_lasso, atol=1e-6), "alpha2=0 must recover pure Lasso exactly"
print("✅ Exercise 5 passed")

<details>
<summary>💡 Show solution</summary>

```python
def elastic_net_gradient_descent(X, y, alpha1=0.1, alpha2=0.1, learning_rate=0.01, max_iter=1000, tol=1e-4):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0.0

    for _ in range(max_iter):
        y_pred = X @ weights + bias
        error = y_pred - y
        grad_w = (X.T @ error) / n_samples + alpha1 * np.sign(weights) + 2 * alpha2 * weights
        grad_b = np.sum(error) / n_samples

        weights -= learning_rate * grad_w
        bias -= learning_rate * grad_b

        if np.linalg.norm(grad_w, ord=1) < tol:
            break

    return weights, bias
```

</details>